# MLP MultiOutput H10

Este notebook entrena un MLP con salida directa de 10 días para la estación Calamar.

Exporta métricas, predicciones del test externo, diagnóstico de residuos, curvas de aprendizaje e historial de entrenamiento.


In [ ]:

# ============================================================
# MLP MultiOutput H10 + timeseries-cv GroupKFold (train/validation)
# Predicción directa de 10 días de nivel en Calamar
#
# Metodología:
# - Test externo final: últimos 10 registros del CSV.
# - Validación cruzada interna: split_train_val_groupKFold.
# - Modelo: MLP Keras con salida Dense(10), no recursivo.
# - Ventanas de entrada: 30 y 60 días.
# - Exporta métricas, predicciones, residuos, curvas de aprendizaje,
#   historial de entrenamiento y modelo final.
# ============================================================

import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tsxv.splitTrainVal import split_train_val_groupKFold

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from statsmodels.tsa.stattools import bds
from statsmodels.graphics.tsaplots import plot_acf

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

warnings.filterwarnings("ignore")
tf.get_logger().setLevel("ERROR")

# Opcional: evita que TensorFlow intente usar GPU si no tienes CUDA configurado.
try:
    tf.config.set_visible_devices([], "GPU")
except Exception:
    pass


# ============================================================
# 1. Configuración general
# ============================================================

ruta = Path(
    r"C:\Users\Daniel Rangel\Documents\MachineLearning\ProyectoFinal\niveles_corregido_15042026.csv"
)

carpeta_salida = Path(
    r"C:\Users\Daniel Rangel\Documents\MachineLearning\ProyectoFinal\ResultadosMLP_MultiOutput_H10"
)
carpeta_salida.mkdir(parents=True, exist_ok=True)

# Archivos tabulares
archivo_resultados = carpeta_salida / "resultados_mlp_multioutput_h10_timeseries_cv_bds.csv"
archivo_resumen = carpeta_salida / "resumen_mlp_multioutput_h10_timeseries_cv_bds.csv"
archivo_test_final = carpeta_salida / "test_final_externo_mlp_multioutput_h10_calamar.csv"
archivo_metricas_horizontes = carpeta_salida / "metricas_horizontes_1_5_10_mlp_multioutput_h10.csv"
archivo_historial_entrenamiento = carpeta_salida / "historial_entrenamiento_mlp_multioutput_h10.csv"
archivo_historial_modelo_final = carpeta_salida / "historial_modelo_final_mlp_multioutput_h10.csv"

# Figuras
archivo_figura_test = carpeta_salida / "mlp_multioutput_h10_test_final_externo.png"
archivo_acf_residuos = carpeta_salida / "acf_residuos_mlp_multioutput_h10_test_externo.png"
archivo_hist_residuos = carpeta_salida / "histograma_residuos_mlp_multioutput_h10_test_externo.png"
archivo_curva_aprendizaje_cv = carpeta_salida / "curva_aprendizaje_mlp_multioutput_h10_mejor_fold.png"
archivo_curva_aprendizaje_final = carpeta_salida / "curva_aprendizaje_modelo_final_mlp_multioutput_h10.png"

# Modelo y metadata
archivo_modelo = carpeta_salida / "modelo_mlp_multioutput_h10_calamar.keras"
archivo_metadata = carpeta_salida / "metadata_modelo_mlp_multioutput_h10_calamar.json"

# Ventanas de entrada
numInputs_list = [30, 60]

# Horizonte directo de predicción
numOutputs = 10
numJumps = 1
H_test = 10
horizontes_eval = [1, 5, 10]

# BDS
alpha_bds = 0.05

# Reproducibilidad
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

# Hiperparámetros fijos del MLP
hidden_layers_fijo = (64, 32)
dropout_fijo = 0.10
learning_rate_fijo = 0.001
activation_fijo = "relu"

epochs = 100
batch_size = 32
patience = 15
early_stopping_monitor = "val_loss"


# ============================================================
# 2. Funciones auxiliares
# ============================================================

def calcular_metricas(y_real, y_pred):
    mae = mean_absolute_error(y_real, y_pred)
    mse = mean_squared_error(y_real, y_pred)
    return mae, mse


def calcular_metricas_horizontes(y_real, y_pred, horizontes):
    """
    Calcula métricas acumuladas hasta cada horizonte.
    h=5 evalúa los días 1, 2, 3, 4 y 5.
    """
    y_real = np.asarray(y_real).astype(float).ravel()
    y_pred = np.asarray(y_pred).astype(float).ravel()

    resultados = []
    for h in horizontes:
        y_h = y_real[:h]
        yhat_h = y_pred[:h]

        mae = mean_absolute_error(y_h, yhat_h)
        mse = mean_squared_error(y_h, yhat_h)
        rmse = np.sqrt(mse)
        r2 = r2_score(y_h, yhat_h) if len(y_h) >= 2 else np.nan

        mask = y_h != 0
        if np.any(mask):
            mape = np.mean(np.abs((y_h[mask] - yhat_h[mask]) / y_h[mask])) * 100
        else:
            mape = np.nan

        resultados.append({
            "horizonte": h,
            "n_datos": len(y_h),
            "MAE": mae,
            "MSE": mse,
            "RMSE": rmse,
            "R2": r2,
            "MAPE_pct": mape
        })

    return resultados


def calcular_bds_pvalue(residuos, max_dim=2, alpha=0.05):
    """
    Aplica BDS sobre residuos aplanados.
    En salida multioutput, residuos tiene forma n_muestras x horizonte.
    """
    residuos = np.asarray(residuos).astype(float).ravel()
    residuos = residuos[~np.isnan(residuos)]

    if len(residuos) < 20:
        return np.nan, False

    try:
        _, pvalue = bds(residuos, max_dim=max_dim)
        pvalue = np.asarray(pvalue).astype(float)
        pvalue_min = np.nanmin(pvalue)
        pasa_bds = bool(pvalue_min >= alpha)
        return pvalue_min, pasa_bds
    except Exception:
        return np.nan, False


def crear_ventanas_supervisadas(sequence, num_inputs, num_outputs=10, num_jumps=1):
    """
    Crea ventanas supervisadas multioutput.
    X: últimos num_inputs valores.
    y: próximos num_outputs valores.
    """
    sequence = np.asarray(sequence).astype(float)
    X_all = []
    y_all = []

    max_start = len(sequence) - num_inputs - num_outputs + 1

    for start in range(0, max_start, num_jumps):
        end_input = start + num_inputs
        end_output = end_input + num_outputs
        X_all.append(sequence[start:end_input])
        y_all.append(sequence[end_input:end_output])

    return np.asarray(X_all), np.asarray(y_all)


def escalar_y(y_train):
    """
    Escala y usando solo el conjunto de entrenamiento del fold.
    Se usa un único promedio y desviación para toda la matriz y_train.
    """
    y_train = np.asarray(y_train).astype(float)
    y_mean = np.mean(y_train)
    y_std = np.std(y_train)

    if y_std == 0:
        y_std = 1.0

    y_train_scaled = (y_train - y_mean) / y_std
    return y_train_scaled, y_mean, y_std


def transformar_y(y, y_mean, y_std):
    y = np.asarray(y).astype(float)
    return (y - y_mean) / y_std


def invertir_y(y_scaled, y_mean, y_std):
    y_scaled = np.asarray(y_scaled).astype(float)
    return y_scaled * y_std + y_mean


def crear_modelo_mlp(num_inputs, num_outputs, X_train_para_normalizar):
    """
    Crea MLP Keras MultiOutput.
    - Normalization se adapta solo con X_train del fold.
    - Dense(num_outputs) predice los 10 días de una vez.
    """
    normalizer = layers.Normalization(axis=-1)
    normalizer.adapt(np.asarray(X_train_para_normalizar).astype(float))

    entrada = layers.Input(shape=(num_inputs,))
    x = normalizer(entrada)

    for units in hidden_layers_fijo:
        x = layers.Dense(units, activation=activation_fijo)(x)
        if dropout_fijo > 0:
            x = layers.Dropout(dropout_fijo)(x)

    salida = layers.Dense(num_outputs)(x)

    model = models.Model(inputs=entrada, outputs=salida)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate_fijo),
        loss="mse",
        metrics=["mae", "mse"]
    )

    return model


def convertir_a_serializable(obj):
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.ndarray,)):
        return obj.tolist()
    return obj


def dataframe_historial(history, contexto):
    """
    Convierte history.history de Keras en DataFrame con metadatos del fold/modelo.
    """
    n_epochs = len(history.history["loss"])

    df_hist = pd.DataFrame({
        "epoch": np.arange(1, n_epochs + 1),
        "loss_train": history.history.get("loss", [np.nan] * n_epochs),
        "loss_val": history.history.get("val_loss", [np.nan] * n_epochs),
        "mae_train": history.history.get("mae", [np.nan] * n_epochs),
        "mae_val": history.history.get("val_mae", [np.nan] * n_epochs),
        "mse_train": history.history.get("mse", [np.nan] * n_epochs),
        "mse_val": history.history.get("val_mse", [np.nan] * n_epochs),
    })

    for k, v in contexto.items():
        df_hist[k] = v

    df_hist["epochs_usados"] = n_epochs
    return df_hist


def graficar_curva_aprendizaje(df_hist, archivo_figura, titulo):
    """
    Grafica loss_train y, si existe, loss_val.
    """
    plt.figure(figsize=(9, 5))
    plt.plot(df_hist["epoch"], df_hist["loss_train"], label="Loss train", linewidth=2)

    if "loss_val" in df_hist.columns and df_hist["loss_val"].notna().any():
        plt.plot(df_hist["epoch"], df_hist["loss_val"], label="Loss validation", linewidth=2, linestyle="--")

    plt.xlabel("Época")
    plt.ylabel("Loss")
    plt.title(titulo)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(archivo_figura, dpi=200)
    plt.show()
    plt.close()


# ============================================================
# 3. Cargar datos
# ============================================================

if not ruta.exists():
    raise FileNotFoundError(
        f"No se encontró el archivo de datos en:\n{ruta}\n\n"
        "Ajusta la variable ruta al inicio del notebook."
    )

df = pd.read_csv(ruta)
df["Fecha"] = pd.to_datetime(df["Fecha"], dayfirst=True, errors="coerce")
df = df.dropna(subset=["Fecha", "Calamar"])
df = df.sort_values("Fecha").reset_index(drop=True)

print("=" * 80)
print("DATOS CARGADOS")
print("=" * 80)
print("Dimensión del dataset:", df.shape)
print("Rango temporal:", df["Fecha"].min(), "->", df["Fecha"].max())


# ============================================================
# 4. Separar test externo final: últimos 10 registros
# ============================================================

if len(df) <= H_test + max(numInputs_list) + numOutputs:
    raise ValueError(
        "La serie no tiene suficientes datos para separar test externo "
        "y construir ventanas supervisadas."
    )

df_trainval = df.iloc[:-H_test].copy()
df_test_ext = df.iloc[-H_test:].copy()
sequence_trainval = df_trainval["Calamar"].values.astype(float)

print("\n" + "=" * 80)
print("SEPARACIÓN TEMPORAL")
print("=" * 80)
print("Train/validación interna:", df_trainval["Fecha"].min(), "->", df_trainval["Fecha"].max())
print("Test externo final      :", df_test_ext["Fecha"].min(), "->", df_test_ext["Fecha"].max())
print("Muestras train/val:", len(df_trainval))
print("Muestras test externo:", len(df_test_ext))


# ============================================================
# 5. Validación cruzada temporal con MLP MultiOutput
# ============================================================

resultados = []
historiales_entrenamiento = []

print("\n" + "=" * 80)
print("CONFIGURACIÓN DE BÚSQUEDA")
print("=" * 80)
print("Modelo: MLP Keras MultiOutput")
print("Ventanas evaluadas:", numInputs_list)
print("Horizonte multioutput:", numOutputs)
print("Parámetros fijos:", {
    "hidden_layers": hidden_layers_fijo,
    "dropout": dropout_fijo,
    "learning_rate": learning_rate_fijo,
    "epochs_max": epochs,
    "batch_size": batch_size,
    "patience": patience
})

for numInputs in numInputs_list:
    print("\n" + "-" * 80)
    print(f"Ventana de entrada: {numInputs} días")
    print("-" * 80)

    X, y, Xcv, ycv = split_train_val_groupKFold(
        sequence_trainval,
        numInputs,
        numOutputs,
        numJumps
    )

    print("Número de folds generados:", len(X))

    mae_val_folds = []
    mse_val_folds = []
    bds_pvalues = []
    bds_flags = []

    for fold in sorted(X.keys()):
        tf.keras.backend.clear_session()
        tf.random.set_seed(RANDOM_STATE)

        X_train_fold = np.asarray(X[fold]).astype(float)
        y_train_fold = np.asarray(y[fold]).astype(float)

        X_val_fold = np.asarray(Xcv[fold]).astype(float)
        y_val_fold = np.asarray(ycv[fold]).astype(float)

        y_train_scaled, y_mean_fold, y_std_fold = escalar_y(y_train_fold)
        y_val_scaled = transformar_y(y_val_fold, y_mean_fold, y_std_fold)

        model = crear_modelo_mlp(
            num_inputs=numInputs,
            num_outputs=numOutputs,
            X_train_para_normalizar=X_train_fold
        )

        early_stop = callbacks.EarlyStopping(
            monitor=early_stopping_monitor,
            patience=patience,
            restore_best_weights=True
        )

        history = model.fit(
            X_train_fold,
            y_train_scaled,
            validation_data=(X_val_fold, y_val_scaled),
            epochs=epochs,
            batch_size=batch_size,
            callbacks=[early_stop],
            verbose=0,
            shuffle=False
        )

        contexto_hist = {
            "tipo_entrenamiento": "validacion_cruzada",
            "numInputs": numInputs,
            "fold": fold,
            "hidden_layers": str(hidden_layers_fijo),
            "dropout": dropout_fijo,
            "learning_rate": learning_rate_fijo,
            "patience": patience,
            "epochs_max": epochs,
            "early_stopping_monitor": early_stopping_monitor,
            "y_mean": y_mean_fold,
            "y_std": y_std_fold,
        }
        historiales_entrenamiento.append(
            dataframe_historial(history, contexto_hist)
        )

        y_val_pred_scaled = model.predict(X_val_fold, verbose=0)
        y_val_pred = invertir_y(y_val_pred_scaled, y_mean_fold, y_std_fold)
        residuos_val = y_val_fold - y_val_pred

        mae_val, mse_val = calcular_metricas(y_val_fold.ravel(), y_val_pred.ravel())
        pvalue_bds, pasa_bds = calcular_bds_pvalue(
            residuos_val,
            max_dim=2,
            alpha=alpha_bds
        )

        mae_val_folds.append(mae_val)
        mse_val_folds.append(mse_val)
        bds_pvalues.append(pvalue_bds)
        bds_flags.append(pasa_bds)

        print(
            f"Fold {fold} | ventana={numInputs} | "
            f"epochs usados={len(history.history['loss'])} | "
            f"MAE_val_H10={mae_val:.4f} | MSE_val_H10={mse_val:.4f} | "
            f"BDS_pvalue={pvalue_bds:.4f} | pasa_BDS={pasa_bds}"
        )

    fila = {
        "numInputs": numInputs,
        "ventana": f"{numInputs} días",
        "numOutputs": numOutputs,
        "numJumps": numJumps,
        "hidden_layers": str(hidden_layers_fijo),
        "dropout": dropout_fijo,
        "learning_rate": learning_rate_fijo,
        "activation": activation_fijo,
        "epochs_max": epochs,
        "batch_size": batch_size,
        "patience": patience,
        "MAE_val_h10_mean": np.mean(mae_val_folds),
        "MAE_val_h10_std": np.std(mae_val_folds),
        "MSE_val_h10_mean": np.mean(mse_val_folds),
        "MSE_val_h10_std": np.std(mse_val_folds),
        "BDS_pvalue_mean": np.nanmean(bds_pvalues),
        "BDS_pvalue_min": np.nanmin(bds_pvalues),
        "BDS_folds_pass": int(np.sum(bds_flags)),
        "BDS_all_folds_pass": bool(np.all(bds_flags)),
        "BDS_any_fold_pass": bool(np.any(bds_flags))
    }

    resultados.append(fila)
    print(f"Ventana {numInputs} días terminada.")


# ============================================================
# 6. Guardar resultados de validación e historial
# ============================================================

resultados_df = pd.DataFrame(resultados)
resultados_df.to_csv(archivo_resultados, index=False, encoding="utf-8-sig")

historial_entrenamiento_df = pd.concat(historiales_entrenamiento, ignore_index=True)
historial_entrenamiento_df.to_csv(
    archivo_historial_entrenamiento,
    index=False,
    encoding="utf-8-sig"
)

print("\n" + "=" * 80)
print("VALIDACIÓN TERMINADA")
print("=" * 80)
print("Resultados completos guardados en:", archivo_resultados)
print("Historial de entrenamiento guardado en:", archivo_historial_entrenamiento)


# ============================================================
# 7. Selección del mejor modelo con MAE H10 + BDS
# ============================================================

candidatos_bds = resultados_df[resultados_df["BDS_all_folds_pass"] == True].copy()

if len(candidatos_bds) > 0:
    criterio_bds_usado = "BDS_all_folds_pass"
    base_seleccion = candidatos_bds
else:
    criterio_bds_usado = "max_BDS_folds_pass"
    base_seleccion = resultados_df.sort_values(
        ["BDS_folds_pass", "MAE_val_h10_mean", "MSE_val_h10_mean"],
        ascending=[False, True, True]
    )

mejor_mae = (
    base_seleccion
    .sort_values(["MAE_val_h10_mean", "MSE_val_h10_mean"], ascending=True)
    .iloc[0]
)

mejor_mse = (
    base_seleccion
    .sort_values(["MSE_val_h10_mean", "MAE_val_h10_mean"], ascending=True)
    .iloc[0]
)

resumen_seleccion = pd.DataFrame([
    {
        "criterio": "MAE_h10",
        "criterio_bds_usado": criterio_bds_usado,
        **mejor_mae.to_dict()
    },
    {
        "criterio": "MSE_h10",
        "criterio_bds_usado": criterio_bds_usado,
        **mejor_mse.to_dict()
    }
])

resumen_seleccion.to_csv(archivo_resumen, index=False, encoding="utf-8-sig")

print("\n" + "=" * 80)
print("MEJORES CONFIGURACIONES")
print("=" * 80)
print(resumen_seleccion[[
    "criterio",
    "criterio_bds_usado",
    "numInputs",
    "ventana",
    "hidden_layers",
    "dropout",
    "learning_rate",
    "MAE_val_h10_mean",
    "MSE_val_h10_mean",
    "BDS_pvalue_min",
    "BDS_folds_pass",
    "BDS_all_folds_pass"
]])

modelo_final_row = mejor_mae.copy()
best_numInputs = int(modelo_final_row["numInputs"])

print("\n" + "=" * 80)
print("MODELO FINAL SELECCIONADO")
print("=" * 80)
print(modelo_final_row[[
    "numInputs",
    "ventana",
    "hidden_layers",
    "dropout",
    "learning_rate",
    "MAE_val_h10_mean",
    "MSE_val_h10_mean",
    "BDS_pvalue_min",
    "BDS_folds_pass",
    "BDS_all_folds_pass"
]])


# ============================================================
# 8. Curva de aprendizaje de un fold representativo
# ============================================================

hist_best_window = historial_entrenamiento_df[
    historial_entrenamiento_df["numInputs"] == best_numInputs
].copy()

# Se escoge como fold representativo el que tuvo menor val_loss mínimo.
resumen_hist = (
    hist_best_window
    .groupby("fold", as_index=False)["loss_val"]
    .min()
    .sort_values("loss_val", ascending=True)
)

fold_representativo = int(resumen_hist.iloc[0]["fold"])

hist_fold_plot = hist_best_window[
    hist_best_window["fold"] == fold_representativo
].copy()

graficar_curva_aprendizaje(
    df_hist=hist_fold_plot,
    archivo_figura=archivo_curva_aprendizaje_cv,
    titulo=(
        f"Curva de aprendizaje MLP MultiOutput H10 | "
        f"ventana={best_numInputs} | fold={fold_representativo}"
    )
)

print("Curva de aprendizaje CV guardada en:", archivo_curva_aprendizaje_cv)


# ============================================================
# 9. Entrenar modelo final sin incluir test externo
# ============================================================

X_trainval_all, y_trainval_all = crear_ventanas_supervisadas(
    sequence_trainval,
    num_inputs=best_numInputs,
    num_outputs=numOutputs,
    num_jumps=numJumps
)

y_trainval_scaled, y_mean_final, y_std_final = escalar_y(y_trainval_all)

modelo_final = crear_modelo_mlp(
    num_inputs=best_numInputs,
    num_outputs=numOutputs,
    X_train_para_normalizar=X_trainval_all
)

# El modelo final se entrena con todo trainval.
# Por eso se monitorea loss y no val_loss.
early_stop_final = callbacks.EarlyStopping(
    monitor="loss",
    patience=patience,
    restore_best_weights=True
)

history_final = modelo_final.fit(
    X_trainval_all,
    y_trainval_scaled,
    epochs=epochs,
    batch_size=batch_size,
    callbacks=[early_stop_final],
    verbose=1,
    shuffle=False
)

historial_final_df = dataframe_historial(
    history_final,
    contexto={
        "tipo_entrenamiento": "modelo_final",
        "numInputs": best_numInputs,
        "fold": "final",
        "hidden_layers": str(hidden_layers_fijo),
        "dropout": dropout_fijo,
        "learning_rate": learning_rate_fijo,
        "patience": patience,
        "epochs_max": epochs,
        "early_stopping_monitor": "loss",
        "y_mean": y_mean_final,
        "y_std": y_std_final,
    }
)

historial_final_df.to_csv(
    archivo_historial_modelo_final,
    index=False,
    encoding="utf-8-sig"
)

graficar_curva_aprendizaje(
    df_hist=historial_final_df,
    archivo_figura=archivo_curva_aprendizaje_final,
    titulo=f"Curva de aprendizaje modelo final MLP MultiOutput H10 | ventana={best_numInputs}"
)

print("Historial modelo final guardado en:", archivo_historial_modelo_final)
print("Curva modelo final guardada en:", archivo_curva_aprendizaje_final)


# ============================================================
# 10. Evaluar test externo final: últimos 10 registros
# ============================================================

X_test_ext_final = df_trainval["Calamar"].values.astype(float)[-best_numInputs:].reshape(1, -1)
y_test_ext = df_test_ext["Calamar"].values.astype(float)
fechas_test_ext = df_test_ext["Fecha"].values

y_test_ext_pred_scaled = modelo_final.predict(X_test_ext_final, verbose=0).ravel()
y_test_ext_pred = invertir_y(y_test_ext_pred_scaled, y_mean_final, y_std_final).ravel()
residuos_test_ext = y_test_ext - y_test_ext_pred

mae_test_ext, mse_test_ext = calcular_metricas(y_test_ext, y_test_ext_pred)

metricas_horizontes = calcular_metricas_horizontes(
    y_real=y_test_ext,
    y_pred=y_test_ext_pred,
    horizontes=horizontes_eval
)

df_metricas_horizontes = pd.DataFrame(metricas_horizontes)
df_metricas_horizontes.to_csv(archivo_metricas_horizontes, index=False, encoding="utf-8-sig")

print("\n" + "=" * 80)
print("EVALUACIÓN FINAL EN TEST EXTERNO H10")
print("=" * 80)
print("Periodo test externo:", pd.to_datetime(fechas_test_ext).min(), "->", pd.to_datetime(fechas_test_ext).max())
print(f"MAE test externo H10: {mae_test_ext:.4f}")
print(f"MSE test externo H10: {mse_test_ext:.4f}")
print("\nMétricas por horizonte acumulado:")
print(df_metricas_horizontes)


df_test_final = pd.DataFrame({
    "Fecha": pd.to_datetime(fechas_test_ext),
    "horizonte": np.arange(1, numOutputs + 1),
    "Calamar_real": y_test_ext,
    "Calamar_predicho": y_test_ext_pred,
    "Residuo": residuos_test_ext
})

df_test_final.to_csv(archivo_test_final, index=False, encoding="utf-8-sig")


# ============================================================
# 11. Diagnóstico de residuos en test externo
# ============================================================

plt.figure(figsize=(9, 5))
plt.hist(df_test_final["Residuo"], bins=min(10, len(df_test_final)), edgecolor="black", alpha=0.75)
plt.axvline(0, color="red", linestyle="--", linewidth=1.5)
plt.xlabel("Residuo")
plt.ylabel("Frecuencia")
plt.title("Histograma de residuos - MLP MultiOutput H10 en test externo")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(archivo_hist_residuos, dpi=200)
plt.show()
plt.close()

residuos_acf = df_test_final["Residuo"].dropna()
max_lags = min(9, len(residuos_acf) - 1)

if max_lags >= 1:
    fig, ax = plt.subplots(figsize=(10, 5))
    plot_acf(residuos_acf, lags=max_lags, ax=ax)
    ax.set_title("ACF de residuos - MLP MultiOutput H10 en test externo")
    ax.set_xlabel("Rezago")
    ax.set_ylabel("Autocorrelación")
    plt.tight_layout()
    plt.savefig(archivo_acf_residuos, dpi=200)
    plt.show()
    plt.close(fig)
else:
    print("No hay suficientes residuos para graficar la ACF.")


# ============================================================
# 12. Figura test externo: 10 predicciones vs 10 reales
# ============================================================

plt.figure(figsize=(12, 5))
plt.plot(
    df_test_final["Fecha"],
    df_test_final["Calamar_real"],
    label="Calamar real",
    linewidth=2.0,
    marker="o"
)
plt.plot(
    df_test_final["Fecha"],
    df_test_final["Calamar_predicho"],
    label="Calamar predicho - MLP",
    linewidth=2.0,
    linestyle="--",
    marker="o"
)
plt.xlabel("Fecha")
plt.ylabel("Nivel en Calamar")
plt.title(
    f"MLP MultiOutput - Test externo final H10 | "
    f"MAE={mae_test_ext:.2f}, MSE={mse_test_ext:.2f}"
)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(archivo_figura_test, dpi=200)
plt.show()
plt.close()
print(f"Figura test externo guardada: {archivo_figura_test}")


# ============================================================
# 13. Guardar modelo y metadata
# ============================================================

modelo_final.save(archivo_modelo)

best_params = {
    "hidden_layers": str(hidden_layers_fijo),
    "dropout": float(dropout_fijo),
    "learning_rate": float(learning_rate_fijo),
    "activation": activation_fijo,
    "epochs_max": int(epochs),
    "batch_size": int(batch_size),
    "patience": int(patience)
}

metadata = {
    "target": "Calamar",
    "numInputs_seleccionado": best_numInputs,
    "numOutputs": numOutputs,
    "numJumps": numJumps,
    "ventanas_evaluadas": numInputs_list,
    "horizontes_eval_test": horizontes_eval,
    "best_params": best_params,
    "modelo": "Keras MLP MultiOutput con capa Normalization y salida Dense(10)",
    "criterio_final": (
        "Menor MAE de validación acumulado sobre la salida completa H10, "
        "usando BDS como criterio de diagnóstico/filtro. "
        "El modelo final fue entrenado sin incluir los últimos 10 registros, "
        "reservados como test externo final."
    ),
    "validacion_cruzada": "split_train_val_groupKFold de timeseries-cv",
    "y_scaler": {
        "mean": float(y_mean_final),
        "std": float(y_std_final)
    },
    "fecha_inicio_trainval": str(df_trainval["Fecha"].min()),
    "fecha_fin_trainval": str(df_trainval["Fecha"].max()),
    "fecha_inicio_test_externo": str(df_test_ext["Fecha"].min()),
    "fecha_fin_test_externo": str(df_test_ext["Fecha"].max()),
    "MAE_test_externo": float(mae_test_ext),
    "MSE_test_externo": float(mse_test_ext),
    "archivo_resultados": str(archivo_resultados),
    "archivo_resumen": str(archivo_resumen),
    "archivo_test_final": str(archivo_test_final),
    "archivo_metricas_horizontes": str(archivo_metricas_horizontes),
    "archivo_historial_entrenamiento": str(archivo_historial_entrenamiento),
    "archivo_historial_modelo_final": str(archivo_historial_modelo_final),
    "archivo_modelo": str(archivo_modelo),
    "archivo_figura_test": str(archivo_figura_test),
    "archivo_acf_residuos": str(archivo_acf_residuos),
    "archivo_hist_residuos": str(archivo_hist_residuos),
    "archivo_curva_aprendizaje_cv": str(archivo_curva_aprendizaje_cv),
    "archivo_curva_aprendizaje_final": str(archivo_curva_aprendizaje_final),
    "selection_summary": {
        k: convertir_a_serializable(v)
        for k, v in modelo_final_row.to_dict().items()
    }
}

with open(archivo_metadata, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=4, ensure_ascii=False)


# ============================================================
# 14. Mensaje final
# ============================================================

print("\n" + "=" * 80)
print("ARCHIVOS GUARDADOS")
print("=" * 80)
print("Resultados completos        :", archivo_resultados)
print("Resumen selección           :", archivo_resumen)
print("Test externo final          :", archivo_test_final)
print("Métricas horizontes         :", archivo_metricas_horizontes)
print("Historial entrenamiento CV  :", archivo_historial_entrenamiento)
print("Historial modelo final      :", archivo_historial_modelo_final)
print("Modelo Keras                :", archivo_modelo)
print("Metadata JSON               :", archivo_metadata)
print("Figura test externo         :", archivo_figura_test)
print("ACF residuos                :", archivo_acf_residuos)
print("Histograma residuos         :", archivo_hist_residuos)
print("Curva aprendizaje CV        :", archivo_curva_aprendizaje_cv)
print("Curva aprendizaje final     :", archivo_curva_aprendizaje_final)
